# H2: why do patches act like seeing?
Patching the "icy" state makes the model infer the cause (snow), just as reading "icy" does. **Is part of that inference already inside the patched state**, because the "icy road" representation overlaps with the "snow" representation?

**Test.** At the key word, the patch changes the state by Δ = (true state − base state). We remove from Δ its component along the scenario's **cause direction** Ĉ (mean state when the cause is stated true minus false, at the cause sentence), Δ' = Δ − (Δ·Ĉ)Ĉ, and patch base + Δ' instead.
* **H2 predicts** D rises (towards doing) when the cause component is removed.
* **Control:** remove the component along an *unrelated* scenario's cause direction instead. This removes a similar amount of the state but should not change D.
* Projection removes only the *linear* overlap at that layer, so a null result is weaker evidence than a positive one.

Uses the valid scenarios of both patching runs (exploration + confirmation) and their reference tables. Word patches at layers 0–15. T4, about 25 minutes. **Runtime → Run all.**

In [ ]:
# Mount Google Drive and work inside the see-vs-do folder (story files live there; results are saved there)
import os
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/see-vs-do")
OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 3
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
import glob, matplotlib.pyplot as plt
TAG = "H2"
PATCH_LAYERS = [L for L in LAYERS if L <= 15]
PI = [LAYERS.index(L) for L in PATCH_LAYERS]
FILLERS = ["Was there a cat nearby?", "Did someone in the story have a brother?", "Was a radio playing?"]
PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med", **{f"f{i}": f"q_f{i}" for i in range(len(FILLERS))}}

refs = []
for setname in ["exploration", "confirmation"]:
    r = pd.read_csv(f"{OUT}/P_{setname}_reference_scenarios.csv", index_col=0); r["set"] = setname; refs.append(r)
REF = pd.concat(refs)
VALID = REF.index[REF.valid].tolist()
P = pd.read_csv("pairs_chain.csv")
for c in [c for c in P.columns if c.startswith(("span_", "wspan_"))]:
    P[c] = P[c].apply(ast.literal_eval)
P = P[P.scenario_id.isin(VALID) & (((P.set == "exploration") & P.variant.between(16, 19)) | ((P.set == "confirmation") & P.variant.between(16, 23)))].reset_index(drop=True)
for i, f in enumerate(FILLERS):
    P[f"q_f{i}"] = f
print(f"{len(VALID)} valid scenarios, {len(P)} items, patch layers {PATCH_LAYERS}")

# cause directions from the original story files (cause sentence stated true vs false), training variants 0-15
SC = pd.concat([pd.read_csv("stories_chain.csv"), pd.read_csv("stories_chain_confirm.csv")])
SC["variant"] = SC.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in ["span_cause_t", "span_cause_f"]:
    SC[c] = SC[c].apply(ast.literal_eval)
SC = SC[SC.scenario_id.isin(VALID) & SC.variant.between(0, 15)].reset_index(drop=True)
Ht = key_states(SC.cause_t.tolist(), SC.q_cause.tolist(), SC.span_cause_t.tolist(), LAYERS)
Hf = key_states(SC.cause_f.tolist(), SC.q_cause.tolist(), SC.span_cause_f.tolist(), LAYERS)
CDIR = {}
for s in VALID:
    m = (SC.scenario_id == s).values
    d = Ht[m].mean(0) - Hf[m].mean(0)
    CDIR[s] = d / d.norm(dim=-1, keepdim=True)          # [L, H] unit vectors
OTHER = {s: VALID[(VALID.index(s) + 1) % len(VALID)] for s in VALID}

## Build the projected patches, then patch

In [ ]:
def key_positions(stories, qs, spans, wspans):
    """Per row: index of the last token of the key word, and of the sentence end."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True, return_offsets_mapping=True)
    offs = enc.pop("offset_mapping"); am = enc["attention_mask"]
    wl, se = [], []
    for i, ((a, b), (wa, wb)) in enumerate(zip(spans, wspans)):
        def last_hit(x, y):
            x, y = x + STORY_OFFSET, y + STORY_OFFSET
            h = ((offs[i, :, 0] < y) & (offs[i, :, 1] > x) & (am[i] == 1)).nonzero().flatten()
            assert len(h) > 0
            return h[-1].item()
        wl.append(last_hit(wa, wb)); se.append(last_hit(a, b))
    return enc, wl, se

def record(stories, qs, spans, wspans, bs=BS):
    """States at [key-word-last ... sentence-end] for every layer in LAYERS. Returns list of [L, n_pos, H] tensors."""
    out = []
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        for r in range(len(wl)):
            out.append(torch.stack([hs[L + 1][r, wl[r]:se[r] + 1].float().cpu() for L in LAYERS]))
    return out

def belief_patch(stories, qs, spans, wspans, src, li, mode, bs=BS):
    """Run the target stories with states replaced at layer LAYERS[li]. src[row] = [L, n_pos, H]."""
    res, skipped = [], 0
    L = LAYERS[li]
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        items = []
        for r in range(len(wl)):
            s_states = src[i + r][li]                         # [n_pos_src, H]
            n_tgt = se[r] - wl[r] + 1
            if mode == "word":
                items.append(([wl[r]], s_states[:1]))
            elif s_states.shape[0] == n_tgt:
                items.append((list(range(wl[r], se[r] + 1)), s_states))
            else:
                items.append(None); skipped += 1
        def hook(mod, inp, out, items=items):
            h = _layer_out(out).clone()
            for r, it in enumerate(items):
                if it is not None:
                    h[r, it[0]] = it[1].to(h.device, h.dtype)
            return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
        handle = lm.model.layers[L].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        b = (logp[:, YES] - logp[:, NO]).tolist()
        res += [np.nan if it is None else v for it, v in zip(items, b)]
    return np.array(res), skipped


q0 = P.q_med.tolist()
t0 = time.time()
TRUE = record(P.see.tolist(), q0, P.span_see.tolist(), P.wspan_see.tolist())
BASE = record(P.base.tolist(), q0, P.span_base.tolist(), P.wspan_base.tolist())
SRC = {"true": [], "minus_cause": [], "minus_unrelated": []}
stats = []
for i, s in enumerate(P.scenario_id):
    tr, bs = TRUE[i][:, :1], BASE[i][:, :1]            # [L, 1, H] at the key word
    delta = tr - bs
    out = {"true": tr}
    for kind, u in [("minus_cause", CDIR[s]), ("minus_unrelated", CDIR[OTHER[s]])]:
        proj = (delta[:, 0] * u).sum(-1, keepdim=True) * u      # [L, H]
        out[kind] = bs + (delta[:, 0] - proj)[:, None]
        if kind == "minus_cause":
            share_c = (proj.norm(dim=-1) / delta[:, 0].norm(dim=-1))
        else:
            share_u = (proj.norm(dim=-1) / delta[:, 0].norm(dim=-1))
    for k in SRC:
        SRC[k].append(out[k])
    for li, L in enumerate(LAYERS):
        stats.append(dict(scenario_id=s, layer=L, cos_delta_cause=(share_c[li]).item(), cos_delta_unrel=(share_u[li]).item()))
stats = pd.DataFrame(stats).groupby(["scenario_id", "layer"]).mean().reset_index()
stats.to_csv(f"{OUT}/{TAG}_overlap.csv", index=False)
display(stats.groupby("layer")[["cos_delta_cause", "cos_delta_unrel"]].mean().round(3))
print("|cos| between the patch change and the cause direction (vs an unrelated cause direction).", f"({(time.time() - t0) / 60:.1f} min)")

stories, spans, wspans = P.base.tolist(), P.span_base.tolist(), P.wspan_base.tolist()
base_b = {p: belief(stories, P[q].tolist(), spans) for p, q in PROBES.items()}
for li in PI:
    L = LAYERS[li]
    path = f"{OUT}/{TAG}_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for kind in SRC:
        r = P[["set", "scenario_id", "item_id"]].copy()
        r["layer"], r["kind"] = L, kind
        for p, q in PROBES.items():
            b, _ = belief_patch(stories, P[q].tolist(), spans, wspans, SRC[kind], li, "word")
            r[f"d{p}"] = b - base_b[p]
        rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min)")

## Results
Paired per scenario: D(minus cause) − D(true), and D(minus unrelated) − D(true). H2 predicts the first is positive and larger than the second.

In [ ]:
rng = np.random.default_rng(0)
def boot_median(x, n=2000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/{TAG}_sweep_layer*.csv")))
sw["dfill"] = sw[[f"df{i}" for i in range(len(FILLERS))]].mean(1)
for p in ["cause", "effect", "med"]:
    sw[f"d{p}C"] = sw[f"d{p}"] - sw.dfill
g = sw.groupby(["scenario_id", "layer", "kind"])[["dcauseC", "deffectC", "dmedC", "dfill"]].mean().reset_index()
g = g.merge(REF[["deffectC_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
g["works"] = g.deffectC >= 0.5 * g.deffectC_see
g["D"] = np.where(g.works, (g.r_see - g.dcauseC / g.deffectC) / (g.r_see - g.r_do), np.nan)
g.to_csv(f"{OUT}/{TAG}_scenario_scores.csv", index=False)
w = g.pivot_table(index=["scenario_id", "layer"], columns="kind", values=["D", "dmedC", "dcauseC", "deffectC"]).reset_index()
rows = []
for L, grp in w.groupby("layer"):
    d1 = grp[("D", "minus_cause")] - grp[("D", "true")]
    d2 = grp[("D", "minus_unrelated")] - grp[("D", "true")]
    lo1, hi1 = boot_median(d1); lo2, hi2 = boot_median(d2)
    rows.append(dict(layer=L, n=int(d1.notna().sum()),
                     D_true=grp[("D", "true")].median(), D_minus_cause=grp[("D", "minus_cause")].median(), D_minus_unrel=grp[("D", "minus_unrelated")].median(),
                     shift_cause=d1.median(), shift_cause_lo=lo1, shift_cause_hi=hi1, shift_unrel=d2.median(), shift_unrel_lo=lo2, shift_unrel_hi=hi2,
                     dmedC_true=grp[("dmedC", "true")].mean(), dmedC_minus_cause=grp[("dmedC", "minus_cause")].mean(),
                     dcauseC_true=grp[("dcauseC", "true")].mean(), dcauseC_minus_cause=grp[("dcauseC", "minus_cause")].mean(),
                     deffectC_true=grp[("deffectC", "true")].mean(), deffectC_minus_cause=grp[("deffectC", "minus_cause")].mean()))
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/{TAG}_summary.csv", index=False)
display(res.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
for col, lo, hi, lab, c in [("shift_cause", "shift_cause_lo", "shift_cause_hi", "cause component removed", "C0"),
                            ("shift_unrel", "shift_unrel_lo", "shift_unrel_hi", "unrelated component removed", "C7")]:
    ax.errorbar(res.layer, res[col], yerr=[res[col] - res[lo], res[hi] - res[col]], fmt="o-", color=c, capsize=3, label=lab)
ax.axhline(0, c="k", lw=.5); ax.set_xlabel("layer"); ax.set_ylabel("change in D vs full patch (median, 95% CI)")
ax.set_title("H2: does removing the cause component make patches more doing-like?"); ax.legend(fontsize=8)
plt.savefig(f"{OUT}/{TAG}_results.png", dpi=150, bbox_inches="tight"); plt.show()